# Calculate Expected Events
Determine the likelihood of observing microlensing events of AGNs

## Generic Imports

In [79]:
import numpy as np
from scipy.integrate import quad

## Calculate Optical Depth

In [80]:
import numpy as np
from scipy.integrate import quad
from astropy.cosmology import Planck18
import astropy.units as u

# 1. Physical Constants and Parameters
G = 4.30091e-6    # kpc M_sun^-1 (km/s)^2
c = 3.0e5         # km/s
rho_s = 4.88e6    # M_sun / kpc^3
r_s = 21.5        # kpc
R_sun = 8.0       # kpc
R_vir = 200.0     # kpc - Limit of the MW halo integration

# 2. Setup AGN Source Distance
z_agn = 5.0
d_a_kpc = Planck18.angular_diameter_distance(z_agn).to(u.kpc).value
D_s = d_a_kpc 

l = np.radians(280)
b = np.radians(-33)

def rho_nfw(r):
    # Avoid singularity at r=0
    return rho_s / ((r / r_s + 1e-9) * (1 + r / r_s)**2)

def dist_from_center(d_l):
    x = R_sun - d_l * np.cos(b) * np.cos(l)
    y = d_l * np.cos(b) * np.sin(l)
    z = d_l * np.sin(b)
    return np.sqrt(x**2 + y**2 + z**2)

# 3. Updated Optical Depth Integral
def integrand(d_l):
    r = dist_from_center(d_l)
    # Using (D_s - d_l) / D_s handles the lensing weight correctly
    return rho_nfw(r) * d_l * (D_s - d_l) / D_s

# We integrate only up to R_vir because density beyond the halo is negligible
tau, error = quad(integrand, 0, R_vir)
tau = (4 * np.pi * G / c**2) * tau

print(f"Optical Depth (tau): {tau:.4e}")

Optical Depth (tau): 9.2692e-07


## Calculate Einstein Radius

In [81]:
import numpy as np
from astropy.cosmology import Planck18
import astropy.units as u

# Constants
G = 6.67430e-11 
c = 2.99792e8    
M_sun = 1.989e30 

# Lens properties
m_lens = 10**4 * M_sun
d_l_kpc = 25.0  # Current lens distance
d_l_m = d_l_kpc * 3.086e19

# Source properties (z=5)
d_s_m = Planck18.angular_diameter_distance(5.0).to(u.m).value
# D_LS in flat cosmology is D_S - D_L
d_ls_m = d_s_m - d_l_m

def einstein_rad(m, d_l, d_s):
    d_ls = d_s - d_l
    # theta_E = sqrt( (4*G*M / c^2) * (D_LS / (D_L * D_S)) )
    theta_E_sq = (4 * G * m / c**2) * (d_ls / (d_l * d_s))
    return np.sqrt(theta_E_sq) # Returns radians

# 1. Angular Einstein Radius
theta_E = einstein_rad(m_lens, d_l_m, d_s_m)
print(f"Einstein radius (radians): {theta_E:.3e}")

# 2. Physical Einstein Radius (R_E = theta_E * D_L)
R_E_m = theta_E * d_l_m
print(f"Physical Einstein radius (meters): {R_E_m:.3e}")

Einstein radius (radians): 2.767e-07
Physical Einstein radius (meters): 2.135e+14


## Calculate Crossing Time

In [82]:
v_perp = 2e5  # Example: 200 km/s (converted to m/s)
seconds_in_year = 365.25 * 24 * 3600

def calculate_crossing_time(r_e, v_t):
    """
    Calculates Einstein crossing time in seconds and years.
    r_e: Einstein radius in meters
    v_t: Transverse velocity in m/s
    """
    t_e_seconds = r_e / v_t
    t_e_years = t_e_seconds / seconds_in_year
    return t_e_seconds, t_e_years

t_e_sec, t_e_yr = calculate_crossing_time(R_E_m, v_perp)

print(f"Einstein crossing time: {t_e_sec:.2e} seconds")
print(f"Einstein crossing time: {t_e_yr:.2f} years")

Einstein crossing time: 1.07e+09 seconds
Einstein crossing time: 33.83 years


## Calculate Event Rate 

In [83]:
def calculate_event_rate(tau, t_e):
    """
    Calculates the microlensing event rate (Gamma).
    tau: Optical depth
    t_e: Einstein crossing time in years
    """
    # Formula: Gamma = 2 * tau / (pi * t_E)
    gamma = (2 * tau) / (np.pi * t_e)
    return gamma

gamma_result = calculate_event_rate(tau, t_e_yr)

print(f"Microlensing event rate (events/star/year): {gamma_result * 20000 * 20:.2f} events over 20 years for 20,000 stars")

Microlensing event rate (events/star/year): 0.01 events over 20 years for 20,000 stars
